# Model Benchmark - Bitcoin (BTC) with Raw Media Enrichment

### Objective
Evaluate whether adding two raw media variables, `avg_tone` and `article_count`, improves BTC crash prediction compared with the market feature set alone.

### Problem Definition
Binary classification of bearish reversal events using the BTC market dataset enriched with Bitcoin media data.

### Modeling Scope
1. Load BTC market data and the multi-coin media file
2. Filter Bitcoin media rows only
3. Align both datasets on the daily timeline
4. Keep the selected market features plus the two raw media variables
5. Run the benchmark pipeline and compare PR-AUC-oriented results


In [ ]:
import pandas as pd
from pipelines.ml_benchmark import run_benchmark_pipeline


### 1. Load Data


In [ ]:
df_btc_price = pd.read_csv("../../data/gold/market/btc_usdt_1d_features.csv")
df_btc_media = pd.read_csv("../../data/gold/market_media_all_coins.csv")

# Keep only Bitcoin rows from the multi-coin media file
df_btc_media = df_btc_media[df_btc_media["crypto"].str.lower() == "bitcoin"].copy()
df_btc_media = df_btc_media.rename(columns={"day": "date"})
df_btc_media = df_btc_media[["date", "article_count", "avg_tone"]]

print(f"{df_btc_price.shape[0]} rows and {df_btc_price.shape[1]} columns in the price dataset.")
print(f"{df_btc_media.shape[0]} rows and {df_btc_media.shape[1]} columns in the filtered media dataset.")
df_btc_media.head()


### 2. Align Price and Media Data


In [ ]:
# Reload raw datasets in this cell to avoid stale notebook state.
df_btc_price = pd.read_csv("../data/gold/market/btc_usdt_1d_features.csv")
df_btc_media = pd.read_csv("../data/gold/market_media_all_coins.csv")

df_btc_media = df_btc_media[df_btc_media["crypto"].str.lower() == "bitcoin"].copy()
df_btc_media = df_btc_media.rename(columns={"day": "date"})
df_btc_media = df_btc_media[["date", "article_count", "avg_tone"]]


def convert_media_dates_with_best_offset(raw_dates, min_price_date, max_price_date):
    candidates = [0, 543, 591]
    best_offset = None
    best_dates = None
    best_overlap = -1
    best_valid = -1

    for offset in candidates:
        parsed = []
        for value in raw_dates:
            if pd.isna(value):
                parsed.append(pd.NaT)
                continue

            s = str(value).strip()
            parts = s.split("-")
            if len(parts) != 3:
                parsed.append(pd.NaT)
                continue

            year_str, month, day = parts
            try:
                year = int(year_str) - offset
            except ValueError:
                parsed.append(pd.NaT)
                continue

            parsed.append(pd.to_datetime(f"{year:04d}-{month}-{day}", errors="coerce"))

        parsed = pd.Series(parsed)
        valid = parsed.notna().sum()
        overlap = ((parsed >= min_price_date) & (parsed <= max_price_date)).sum()

        if overlap > best_overlap or (overlap == best_overlap and valid > best_valid):
            best_offset = offset
            best_dates = parsed
            best_overlap = overlap
            best_valid = valid

    return best_dates, best_offset, best_valid, best_overlap


# Parse BTC price dates first to score media date conversions against the real modeling window.
df_btc_price["open_time"] = pd.to_datetime(df_btc_price["open_time"], errors="coerce")
min_price_date = df_btc_price["open_time"].min()
max_price_date = df_btc_price["open_time"].max()

df_btc_media["date"], chosen_offset, valid_dates, overlap_dates = convert_media_dates_with_best_offset(
    df_btc_media["date"],
    min_price_date,
    max_price_date,
)

invalid_dates = df_btc_media["date"].isna().sum()
print(f"Chosen year offset for media dates: {chosen_offset}")
print(f"Valid media dates after parsing: {valid_dates}")
print(f"Overlapping media dates with BTC price range: {overlap_dates}")
print(f"Invalid media dates after parsing: {invalid_dates}")

df_btc_media = (
    df_btc_media
    .dropna(subset=["date"])
    .sort_values("date")
    .drop_duplicates("date", keep="last")
    .reset_index(drop=True)
)

if df_btc_media.empty:
    raise ValueError(
        "The Bitcoin media dataset is empty after date parsing. "
        "The `day` column in market_media_all_coins.csv could not be converted to supported calendar dates."
    )

min_media_date = df_btc_media["date"].min()
max_media_date = df_btc_media["date"].max()

print(f"BTC price range: {min_price_date.date()} to {max_price_date.date()}")
print(f"BTC media range: {min_media_date.date()} to {max_media_date.date()}")

df_btc_media = df_btc_media[
    (df_btc_media["date"] >= min_price_date) &
    (df_btc_media["date"] <= max_price_date)
].copy()
print(f"After filtering to the BTC price range, {df_btc_media.shape[0]} rows remain in the media dataset.")

if df_btc_media.empty:
    raise ValueError(
        f"No overlapping dates between BTC price data ({min_price_date.date()} to {max_price_date.date()}) "
        f"and media data ({min_media_date.date()} to {max_media_date.date()}). "
        "Check the date calendar or source period in market_media_all_coins.csv before merging."
    )

original_dates = set(df_btc_media["date"])

df_btc_media = (
    df_btc_media
    .set_index("date")
    .reindex(df_btc_price["open_time"])
    .rename_axis("date")
    .reset_index()
)

cols_to_fill = ["avg_tone", "article_count"]
df_btc_media["was_added"] = ~df_btc_media["date"].isin(original_dates)
df_btc_media[cols_to_fill] = df_btc_media[cols_to_fill].fillna(0)

print(f"After reindexing, {df_btc_media.shape[0]} rows remain in the media dataset.")
df_added = df_btc_media[df_btc_media["was_added"]]
print(f"{df_added.shape[0]} rows were added by reindexing.")
df_added.head()


### 3. Build the Modeling Dataset


In [ ]:
df_btc_media_and_price = pd.merge(
    df_btc_price,
    df_btc_media,
    left_on="open_time",
    right_on="date",
    how="inner"
)

df_btc_media_and_price.drop(columns=["open_time", "date", "was_added"], inplace=True)

SELECTED_FEATURES = [
    # Momentum
    "return_1d", "return_7d",

    # Volatility
    "volatility_7d", "volatility_30d",

    # Market behavior
    "buy_pressure",

    # Risk
    "drawdown",

    # Trend
    "ma_ratio",

    # Lags
    "lag_return_1d", "lag_return_7d",
    "lag_volatility_7d",
    "lag_buy_pressure",
    "lag_volume_norm",

    # Additional
    "momentum_acc",
    "momentum_volatility",
    "volume",
    "number_of_trades",
    "quote_asset_volume",

    # Raw media enrichment
    "avg_tone",
    "article_count",
]

TARGET = "target"

df_btc_media_and_price = df_btc_media_and_price[SELECTED_FEATURES + [TARGET]]
df_btc_media_and_price.head()


### 4. Check Missing Values


In [ ]:
df_btc_media_and_price.isna().sum()


### 5. Run Model Benchmark


In [ ]:
results = run_benchmark_pipeline(
    df_btc_media_and_price,
    verbose=True,
    plot_confusion=True
)

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values("pr_auc", ascending=False)

results_df.loc["mean_all"] = results_df.mean()
results_df.loc["mean_no_baseline"] = results_df.drop(
    index=["dummy", "logistic"], errors="ignore"
).mean()

results_df["rank"] = results_df["pr_auc"].rank(ascending=False)

print("=== Model Benchmarking Results ===")
results_df


### Final Note

This notebook isolates the modeling step for BTC with raw media enrichment only. It is designed to answer a narrow question: does adding `avg_tone` and `article_count` improve benchmark performance over the BTC market feature set?
